# Day 5 (Tue Aug 18) — Let's build GPT (1h56m) — the main event

He types EVERYTHING live from an empty notebook — so this starter is just the data + targets. Work in tandem: pause when he names a thing, build it, unpause to compare. This notebook = dev scratchpad (his gpt-dev); consolidate into gpt.py at the end (CS336 tests import a .py).

**arc:** read data → encode/decode → batches → bigram baseline (val ~2.5) → the mathematical trick → single head → multi-head → feedforward → blocks + residuals + layernorm → scale up
**targets:** bigram baseline val ~2.5 · final GPT (n_embd 384, 6 layers, 6 heads, block 256, ~10M params) **val ~1.48** on shakespeare chars
**reuse from my week:** embeddings, Linear, cross-entropy, training loop, lr decay, eval-mode discipline, param-count alarm, shape walks — all of it appears again here

In [22]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)

In [23]:
BATCH_SIZE = 32 # how many independent sequences will we process in parallel?
BLOCK_SIZE = 8 # what is the maximum context length for predictions?
TRAINING_STEPS = 10000 
N_EMBEDDINGS = 32
HEAD_SIZE = N_EMBEDDINGS
HEAD_NUMBER = 4
LEARNING_RATE=1e-3
EVAL_ITERS = 200

In [24]:
# input.txt already downloaded (tiny shakespeare, 1,115,394 chars) — no wget needed
with open('input.txt', 'r') as f:
    text = f.read()
print(len(text))
print(text[:200])

1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


In [25]:
unique_text = sorted(set(text))
vocab_size = len(unique_text)
itos= {ch: i for ch, i in enumerate(unique_text)}
stoi= {i: ch for ch, i in enumerate(unique_text)}

encode = lambda s:[stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])


In [26]:
data = torch.tensor(encode(text), dtype=torch.long)

# Split validation, and train data
n = int(0.9 * len(data))
train_data = data[:n] 
val_data = data[n:] 

In [27]:
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - BLOCK_SIZE, (BATCH_SIZE,))
    x = torch.stack([data[i:i+BLOCK_SIZE] for i in ix]) # 
    y = torch.stack([data[i+1:i+BLOCK_SIZE+1] for i in ix])
    return x, y

In [28]:
xb, yb = get_batch('test')

print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(BATCH_SIZE): # batch dimension
    for t in range(BLOCK_SIZE): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([32, 8])
tensor([[ 6,  1, 52, 53, 58,  1, 58, 47],
        [ 6,  1, 54, 50, 39, 52, 58, 43],
        [ 1, 58, 46, 47, 57,  1, 50, 47],
        [ 0, 32, 46, 43, 56, 43,  1, 42],
        [ 1, 47, 57,  1, 58, 46, 43,  1],
        [ 0, 32, 46, 47, 57,  1, 44, 39],
        [58, 61, 39, 56, 42,  1, 44, 39],
        [27, 10,  0, 35, 46, 63,  6,  1],
        [ 7, 42, 39, 63,  6,  1, 52, 53],
        [61,  6,  1, 39, 57,  1, 61, 43],
        [ 1, 54, 50, 39, 45, 59, 43,  1],
        [ 1, 52, 53, 58,  1, 46, 47, 57],
        [47, 50, 50,  1, 46, 39, 60, 43],
        [57, 58, 39, 63,  1, 46, 47, 51],
        [56, 51, 43, 52, 58, 57,  8,  0],
        [52,  8,  0,  0, 19, 27, 26, 38],
        [57, 58,  0, 35, 46, 39, 58,  1],
        [58, 53,  1, 41, 53, 51, 43,  1],
        [ 0, 13, 52, 42,  1, 47, 44,  1],
        [43, 43, 41, 46,  0, 50, 47, 57],
        [33, 15, 20, 21, 27, 10,  0, 37],
        [39, 58, 46, 43, 56,  1, 61, 43],
        [53, 52, 53, 59, 57,  1, 57, 50],
      

In [29]:
t = nn.Embedding(65, 32)
print(t.weight.shape)                       # (65, 32)
print(t(torch.zeros(4, 8, dtype=torch.long)).shape)   # (4, 8, 32)
a = torch.arange(8)
a

torch.Size([65, 32])
torch.Size([4, 8, 32])


tensor([0, 1, 2, 3, 4, 5, 6, 7])

In [30]:
class Head(nn.Module):
    def __init__(self, head_size) -> None:
        super().__init__()
        self.head_size = head_size
        self.key = nn.Linear(N_EMBEDDINGS, head_size, bias=False)
        self.query = nn.Linear(N_EMBEDDINGS, head_size, bias=False)
        self.value = nn.Linear(N_EMBEDDINGS, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(BLOCK_SIZE, BLOCK_SIZE)))
    
    def forward(self, x):
        B, T, C = x.shape

        k = self.key(x)
        v = self.value(x)
        q = self.query(x)

        kt = k.transpose(-1, -2)

        wei = q @ kt
        wei = wei * self.head_size**-0.5 
        # Makes it so that tokens cant talk to future tokens
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)

        self.out = wei @ v
        return self.out

In [31]:
class MultiHeadAttention(nn.Module):
    def __init__(self, head_num, head_size) -> None:
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(head_num)])
    
    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)

In [32]:
class FeedForward(nn.Module):
    def __init__(self, n_embed) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embed, n_embed),
            nn.ReLU()
        )

    def forward(self, x):
        return self.net(x)

In [33]:
class Block(nn.Module):
    def __init__(self, n_embed, n_head) -> None:
        super().__init__()
        head_size = n_embed // n_head
        self.sa_heads = MultiHeadAttention(n_head, head_size)
        self.ffn = FeedForward(n_embed)

    def forward(self, x):
        x = self.sa_heads(x)
        x = self.ffn(x)
        return x

In [34]:
class BigramLanguageModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, N_EMBEDDINGS)
        self.position_embedding_table = nn.Embedding(BLOCK_SIZE, N_EMBEDDINGS)
        self.blocks = nn.Sequential(
            Block(N_EMBEDDINGS, HEAD_NUMBER),
            Block(N_EMBEDDINGS, HEAD_NUMBER),
            Block(N_EMBEDDINGS, HEAD_NUMBER)
        )
        self.llm_head = nn.Linear(N_EMBEDDINGS, vocab_size) # Linear Layer Construction: 32 numbers in, 16 out
    
    def forward(self, x, target=None):
        B, T = x.shape

        token_emedding = self.token_embedding_table(x) # Batch, Time, Embedding dimension, what i am 
        position_embedding = self.position_embedding_table(torch.arange(T)) # Time, Embedding dimension, where i am 
        x = token_emedding + position_embedding
        x = self.blocks(x)
        logit = self.llm_head(x) # batch, Time, Vocab size, passing x as input to the Linear layer

        if target == None:
            loss = None
        else:
            B, T, C = logit.shape
            logit = logit.view(B*T, C)
            target = target.view(B*T)
            loss = F.cross_entropy(logit, target)
        return logit, loss

    def generate(self, x, max_new_tokens):
        for _ in range(max_new_tokens):
            x_cond = x[:, -BLOCK_SIZE:]
            # Calling image.pngthe module runs forward.
            logit, _ = self(x_cond)
            # Only the last position's prediction is new
            logit = logit[:, -1, :] # (B, T, C) → (B, C)
            prob = F.softmax(logit, -1)
            sample = torch.multinomial(prob, num_samples=1)
            x = torch.cat((x, sample), dim=-1)
        return x

In [35]:
bi = BigramLanguageModel()    

x = encode("ROMEO:")
idx = torch.tensor([x])
out = bi.generate(idx, 200)[0].tolist()
decode(out)

"ROMEO:OHmdwPhXYR!Bkj?aMq?eyM-dE3Cg,jSCHXyCW3OA$cGAA:L,UBjvyLgsriCdNwBRszJTUlDnF'e!N?AcHWm:A;kkmBBxTN:KhI'iG3FmdwQKjvzkpdP-L.d,H:$rgaV' LTw.PG:;,pNQsD?PXUfDopQo $'p-;Jdz-C!moFtYXEGkA; KRrdqdf$3!GegdWaWpvK3L?"

In [36]:
@torch.no_grad()
def estimate_loss():
    out = {}
    bi.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(EVAL_ITERS)
        for k in range(EVAL_ITERS):
            X, Y = get_batch(split)
            logits, loss = bi(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    bi.train()
    return out

In [37]:
# training using AdamW optimizer
optimizer = torch.optim.AdamW(bi.parameters(), lr=LEARNING_RATE)

for i in range(TRAINING_STEPS):
    xb, yb = get_batch('train')
    logit, loss = bi(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(estimate_loss())


{'train': tensor(2.1600), 'val': tensor(2.2001)}


## leaderboard — real numbers only (estimate_loss, BATCH_SIZE 32)

| step | train | val | karpathy val |
|---|---|---|---|
| 3 blocks, no residuals | 2.1600 | **2.2001** | 2.31 |
| + residuals & projections | | | 2.08 |
| + layernorm | | | 2.06 |
| + dropout, scaled up | | | **1.48** |

everything measured before this point (2.37 / 2.45 / 2.48 / 2.56 / 2.40 / 2.78 / 2.25) was ONE batch of 4 — noise floor bigger than the effects, not comparable, kept only as the story of why this table exists.

In [38]:
x = encode("ROMEO:")
idx = torch.tensor([x])
out = bi.generate(idx, 200)[0].tolist()
decode(out)

"ROMEO:\nWhe abud shell'l ichinkt Mofir me him.\nWhall gisone baine;\nBucome, sen mosherew, oridend, nike. Hy ju, ifsull, by old mo; thine sos treeatere, bese ner boup to coud more you fukedss fireing not oro-G"

Self attention basics
- We want to find a way for one token to talk with the other
- The simplest way to do that is for the current token to store the average of its vectors plus its prev vectors.
- **Affinity**: how much token in position i wants to hear from token in position j

x: t0 [1, 0]    t1 [2, 2]    t2 [0, 4]    t3 [4, 0]
xbow[0] = [1, 0]                       
xbow[1] = [(1+2)/2, (0+2)/2] = [1.5, 1]                 
xbow[2] = [(1+2+0)/3, (0+2+4)/3] = [1, 2]                            
xbow[3] = [1.75, 1.5]                 

there are multiple ways to do this, for loops, mat mul, and softmax

In [39]:
B, T, C = 4, 8, 32 
x  = torch.rand(B,T,C)

a = torch.ones((T,T))
tril = torch.tril(a)
wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=1)
out = wei @ x
out.shape

torch.Size([4, 8, 32])

In [40]:
# self attention
B, T, C = 4, 8, 32 
x  = torch.rand(B,T,C)

key = nn.Linear(C, HEAD_SIZE, bias=False)
query = nn.Linear(C, HEAD_SIZE, bias=False)
value = nn.Linear(C, HEAD_SIZE, bias=False)

k = key(x) # (4, 8, 32) @ (32, 16) -> (4, 8, 16)
q = query(x) # (4, 8, 32) @ (32, 16) -> (4, 8, 16)
v = value(x) # (4, 8, 32) @ (32, 16) -> (4, 8, 16)

kt = k.transpose(-1,-2) # (4, 16, 8)

wei = q @ kt
wei = wei * HEAD_SIZE ** 0.5

In [41]:
a = torch.ones(T,T)
tril = torch.tril(a)
wei = wei.masked_fill(tril==0, float('-inf'))
wei = F.softmax(wei, dim=-1)
wei[0]

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0243, 0.9757, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0195, 0.7185, 0.2621, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0374, 0.3171, 0.1422, 0.5033, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0102, 0.1177, 0.3127, 0.4759, 0.0835, 0.0000, 0.0000, 0.0000],
        [0.0121, 0.0168, 0.0232, 0.0324, 0.0969, 0.8187, 0.0000, 0.0000],
        [0.0157, 0.1801, 0.0240, 0.2646, 0.1114, 0.3965, 0.0076, 0.0000],
        [0.0060, 0.3131, 0.2162, 0.1452, 0.0049, 0.2773, 0.0315, 0.0057]],
       grad_fn=<SelectBackward0>)

In [42]:
out = wei @ v
out.shape

torch.Size([4, 8, 32])

## observations

single-batch losses (BATCH_SIZE=4, 10k steps, lr 1e-3) — ALL UNRELIABLE, see below:
- bigram, embedding table doubles as output layer (65x65): **2.37**
- + n_embd split (65->32->65 with lm_head): **2.45**
- + position embeddings: **2.48**
- + single self-attention head: **2.56**
- + multi-head, 4 heads x 8: **2.40**

the numbers went the WRONG WAY for three steps straight and i can't conclude anything from them. karpathy reports ~2.40 single head, ~2.28 multi-head. the problem isn't the model, it's the measurement: one random batch of 4 sequences x 8 chars = 32 predictions. that's noisier than the effect i'm trying to see. i have been comparing architectures with a ruler that can't resolve them.

blocking before any further comparison: estimate_loss() averaging over many batches, both splits, eval/train toggled — and BATCH_SIZE 32. every remaining step in the video is worth 0.05-0.2, which is under my current noise floor.

verified regardless of loss: shapes correct end to end, wei rows sum to 1 and lower-triangular, generate crops to BLOCK_SIZE, head_size = n_embd single-head (karpathy 80:22) then C//n_head for multi-head, ModuleList so the heads actually register.